# Retinal Evidence Explorer — free interactive Colab

Upload a fundus JPG/PNG → extract X1–X4 → run the frozen ResNet152 grader → inspect measured structures and Grad-CAM → ask a local Qwen assistant.

**Before starting:** choose Runtime → Change runtime type → GPU. Free GPU availability and runtime duration vary. This notebook starts no public server or tunnel.

You need the supplied **retinal_models_v1.tar.gz** inference bundle (about 1.9 GB). Place it in your own Google Drive before running. The website source does not include large model weights. The bundle contains no patient images or datasets. Review code and use only the provided checksum-verified bundle, since model files include trusted Python serialization.

This is a research demonstration, not a clinical diagnosis. X2 maps are weak proxies; X3 is a lesion-trained encoder and does not return lesion detections. Historical external grading scores require consistent X3 preprocessing reevaluation.

In [ ]:
import os,sys,subprocess,json,hashlib,tarfile,io,base64,shutil,uuid
from pathlib import Path
from IPython.display import display,HTML,Markdown,Image as DisplayImage
from google.colab import drive,files
import ipywidgets as widgets

# Isolated workers cannot use the notebook kernel's inline plotting backend.
os.environ['MPLBACKEND']='Agg'
SOURCE=Path('/content/retinal-source')
if not SOURCE.exists():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','https://github.com/Inamullah-Colab/Inamullah-Colab.github.io.git',str(SOURCE)],check=True)
    subprocess.run(['git','sparse-checkout','set','local/retinal-colab-evidence'],cwd=SOURCE,check=True)
else:
    subprocess.run(['git','pull','--ff-only'],cwd=SOURCE,check=True)
PROJECT=SOURCE/'local/retinal-colab-evidence'
assert (PROJECT/'app/notebook_runner.py').is_file(),'Notebook source package not found.'
print('Notebook source ready:',PROJECT)

## 1. Load and verify the model bundle

Mount your own Drive and set BUNDLE_PATH below. This notebook reads only the specified bundle; it does not automatically copy images or results to Drive. If you share the notebook with someone else, they also need access to an authorized copy of this bundle.

In [ ]:
drive.mount('/content/drive')
BUNDLE_PATH=Path('/content/drive/MyDrive/retinal_models_v1.tar.gz')  # Change this path if needed.
EXPECTED_SHA256='ccef2305685a9c248f357c1133e29aee564bcfd6d2ff0023b7904630aea13963'
assert BUNDLE_PATH.is_file(),'Place the supplied model bundle in Drive and update BUNDLE_PATH.'
h=hashlib.sha256()
with BUNDLE_PATH.open('rb') as f:
    for chunk in iter(lambda:f.read(1048576),b''):h.update(chunk)
assert h.hexdigest()==EXPECTED_SHA256,'Bundle checksum mismatch. Do not load unverified model files.'
DEST=Path('/content/retinal-artifacts');DEST.mkdir(exist_ok=True)
with tarfile.open(BUNDLE_PATH,'r:gz') as archive:
    members=archive.getmembers()
    assert len(members)<5000 and sum(m.size for m in members)<10*1024**3
    for member in members:
        target=(DEST/member.name).resolve()
        assert target.is_relative_to(DEST.resolve()) and member.name.startswith('model_bundle/')
        assert member.isfile() or member.isdir(),'Links and special files are not permitted.'
    archive.extractall(DEST,members=members,filter='data')
BUNDLE=DEST/'model_bundle'
subprocess.run([sys.executable,str(PROJECT/'scripts/verify_bundle.py'),str(BUNDLE)],check=True)
print('Model bundle verified.')

## 2. Install an isolated model environment

The model pipeline uses Python 3.11 and pinned PyTorch, separate from Colab's notebook kernel. Installation and the first LLM download may take several minutes. Disk space and a usable NVIDIA GPU are required.

In [ ]:
subprocess.run([sys.executable,'-m','pip','-q','install','uv'],check=True)
subprocess.run(['uv','python','install','3.11'],check=True)
ENV=Path('/content/retinal-model-env')
if not (ENV/'bin/python').exists():
    subprocess.run(['uv','venv','--python','3.11',str(ENV)],check=True)
PYTHON=str(ENV/'bin/python')
subprocess.run(['uv','pip','install','--python',PYTHON,'torch==2.3.1','torchvision==0.18.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)
subprocess.run(['uv','pip','install','--python',PYTHON,'-r',str(PROJECT/'requirements-colab.txt')],check=True)
subprocess.run([PYTHON,'-c',"import torch; assert torch.cuda.is_available(), 'Select a GPU runtime and retry'; print(torch.cuda.get_device_name(0))"],check=True)
print('Isolated model environment ready.')

## 3. Upload one image and analyze

Use an image you are permitted to process. Maximum 10 MB, 30 megapixels, and minimum 128 pixels per side. AutoMorph can reject low-quality or incomplete-feature images. Processing takes several minutes.

Your image and intermediate results stay in this notebook runtime until cleanup or runtime termination. Nothing is automatically sent to the website backend or published to GitHub.

In [ ]:
if 'CHAT' in globals() and CHAT.poll() is None:
    CHAT.terminate()
    try:CHAT.wait(timeout=15)
    except subprocess.TimeoutExpired:CHAT.kill();CHAT.wait()
if 'CHAT_LOG' in globals():CHAT_LOG.close()

RUN_DIR=Path('/content/retinal-results')/uuid.uuid4().hex
RUN_DIR.mkdir(parents=True)
previous=Path.cwd();os.chdir(RUN_DIR)
try:uploaded=files.upload()
finally:os.chdir(previous)
assert len(uploaded)==1,'Upload one image at a time.'
name,payload=next(iter(uploaded.items()))
assert len(payload)<=10*1024*1024,'Maximum file size is 10 MB.'
from PIL import Image
with Image.open(io.BytesIO(payload)) as image:
    assert image.format in ('JPEG','PNG') and min(image.size)>=128 and image.width*image.height<=30_000_000
    image.convert('RGB').save(RUN_DIR/'input.png')
INPUT=RUN_DIR/'input.png';OUTPUT=RUN_DIR/'analysis'
command=[PYTHON,str(PROJECT/'app/notebook_runner.py'),'--bundle',str(BUNDLE),'--image',str(INPUT),'--output',str(OUTPUT)]
with (RUN_DIR/'runner.log').open('w') as log:
    completed=subprocess.run(command,stdout=log,stderr=subprocess.STDOUT,text=True)
runner_output=(RUN_DIR/'runner.log').read_text(errors='replace')
print(runner_output[-16000:])
if completed.returncode:
    extraction_log=OUTPUT/'extract.log'
    if extraction_log.exists():print('AutoMorph log tail:\n'+extraction_log.read_text(errors='replace')[-12000:])
    raise RuntimeError('Analysis failed. The underlying error is printed above. Do not rerun setup or upload again; inspect runner.log and extract.log first.')
REPORT=json.loads((OUTPUT/'evidence_report.json').read_text())
if REPORT.get('status')=='graded':
    print('Prediction:',REPORT['predicted_grade'],'— research result requiring clinical review.')
    print('Probabilities:',REPORT['class_probabilities'])
else:
    print('No grade produced:',REPORT['reason'])
    print(REPORT.get('next_step',''))

## 4. Inspect the image-specific evidence

- **X1:** vessel morphology, skeleton occupancy, topology and supplementary processed-pixel length.
- **X2:** weak lesion-evidence proxies. The separate Grad-CAM is actual predicted-class image-branch attribution and is not a lesion detector.
- **X3:** lesion-trained contrastive representation; boxes and lesion counts are **not available** from this checkpoint.
- **X4:** all 72 biomarkers with descriptions, including complexity, tortuosity, curvature, density, width and regional structure.

No normal/abnormal thresholds are inferred from the literature. Physical scale is assumed.

In [ ]:
if REPORT.get('status')!='graded':
    print('No grade produced:',REPORT.get('reason','Incomplete measurements.'))
    print('Available measured values:',json.dumps(REPORT.get('available_features',{}),indent=2))
    print('Missing measurements:',REPORT.get('missing_features',[]))
else:
    import html
    EVIDENCE=REPORT['evidence']
    for panel in EVIDENCE['images']:
        display(Markdown('### '+panel['title']))
        display(DisplayImage(data=base64.b64decode(panel['data_url'].split(',',1)[1])))
        display(Markdown(panel['caption']))
    for stream in ['X1','X2','X3','X4']:
        display(Markdown('## '+stream+' — '+EVIDENCE[stream]['role']))
        if stream=='X3':
            display(Markdown(EVIDENCE[stream]['lesion_detection']['reason']))
            values={str(v['index']):v['value'] for v in EVIDENCE[stream]['top_components']}
        else:values={**EVIDENCE[stream]['features'],**EVIDENCE[stream].get('supplemental_measurements',{})}
        definitions={d['feature']:d for d in EVIDENCE[stream].get('definitions',[])}
        rows=''.join('<tr><td>'+html.escape(key)+'</td><td>'+html.escape(f'{value:.7g}')+'</td><td>'+html.escape(definitions.get(key,{}).get('meaning',''))+'</td></tr>' for key,value in values.items())
        display(HTML('<div style="max-height:420px;overflow:auto"><table><tr><th>Feature</th><th>Value</th><th>Meaning</th></tr>'+rows+'</table></div>'))
    for reference in EVIDENCE['references']:
        display(Markdown('['+reference['title']+']('+reference['url']+') — '+reference['summary']+' '+reference['limits']))

## 5. Ask the image-grounded knowledge assistant

The local Qwen model selects relevant, cited facts from 26 curated topics spanning ophthalmology, systemic-health context, oculomics and model interpretation. Specific metric questions display only the relevant measured values. Short follow-ups can use earlier question context. This is retrieval augmentation, not new weight training or comprehensive clinical expertise. No systemic diagnosis is inferred from the DR report.


In [ ]:
if REPORT.get('status')!='graded':
    print('Grade-explanation chat is unavailable because no grade was produced. See the quality or missing-feature report above.')
else:
    if 'CHAT' in globals() and CHAT.poll() is None:
        CHAT.terminate()
        try:CHAT.wait(timeout=15)
        except subprocess.TimeoutExpired:CHAT.kill();CHAT.wait()
    if 'CHAT_LOG' in globals():CHAT_LOG.close()
    
    CHAT_LOG=(RUN_DIR/'chat-model.log').open('w')
    chat_env=os.environ.copy();chat_env['NOTEBOOK_LLM_DEVICE']='cuda'
    chat_env.pop('MODEL_PATH',None)
    CHAT=subprocess.Popen([PYTHON,str(PROJECT/'app/chat_worker.py'),str(OUTPUT/'evidence_report.json')],stdin=subprocess.PIPE,stdout=subprocess.PIPE,stderr=CHAT_LOG,text=True,bufsize=1,env=chat_env)
    ready=json.loads(CHAT.stdout.readline())
    assert ready.get('ready'),ready
    print('Live local model ready:',ready['model'])
    print('Knowledge topics:',ready.get('knowledge_topics',0),'— cited retrieval augmentation; model weights unchanged')
    question=widgets.Textarea(value='What does my fractal dimension mean, and how does it relate to oculomics?',description='Question:',layout=widgets.Layout(width='100%'))
    ask=widgets.Button(description='Ask about my image',button_style='success');answers=widgets.Output()
    def answer_question(_):
        question_text=question.value.strip()
        if not question_text:return
        ask.disabled=True
        try:
            CHAT.stdin.write(json.dumps({'question':question_text})+'\n');CHAT.stdin.flush()
            response=json.loads(CHAT.stdout.readline())
            with answers:
                answers.clear_output(wait=True)
                print('Question:',question_text)
                if response.get('resolved_query',question_text)!=question_text:print('Follow-up context:',response['resolved_query'])
                print(response.get('answer',response.get('error')))
                for reference in response.get('references',[]):display(Markdown('['+reference['title']+']('+reference['url']+')'))
                reference=response.get('reference')
                if reference:display(Markdown('['+reference['title']+']('+reference['url']+')'))
        finally:ask.disabled=False
    ask.on_click(answer_question)
    display(question,ask,answers)

## 6. Download and clean up

Download the JSON report if wanted. It includes measured features, references and visualization images. Keep it private as appropriate. Cleanup stops the local chat and deletes this analysis folder from the runtime; it does not delete copies you explicitly downloaded or saved to Drive.

In [ ]:
files.download(str(OUTPUT/'evidence_report.json'))

In [ ]:
# Run when finished. This deletes only this notebook analysis folder.
if 'CHAT' in globals() and CHAT.poll() is None:
    CHAT.terminate()
    try:CHAT.wait(timeout=15)
    except subprocess.TimeoutExpired:CHAT.kill();CHAT.wait()
if 'CHAT_LOG' in globals():CHAT_LOG.close()
if 'RUN_DIR' in globals() and RUN_DIR.resolve().is_relative_to(Path('/content/retinal-results')):
    shutil.rmtree(RUN_DIR)
print('Local analysis files cleaned up.')